# Model: google/rembert

## 🚀 RMIT Hackathon 2026 — Challenge 1: Multilingual Prompt-Attack Classification
> **Notebook Naming:** `RMIT_SGS_BakhiBachKhoa_C1`
> **BTC Strict Rule:** You must build your solution on strictly ONE approved model backbone. No multi-backbone ensembling or blending.
> **Chosen Single Backbone:** `google/rembert` (32 layers, hidden dimension 1152, ~575M parameters).
> **Checkpoint Strategy:** 5-Fold cross-validation with fold averaging of the same single backbone RemBERT (allowed by competition rules).

---

### 📋 Checklist Tuân Thủ Luật Cuộc Thi (Trước Khi Submit)
- [x] **Single-Model Rule:** Xây dựng trên DUY NHẤT 1 pretrained backbone `google/rembert`. Không blend hay ensemble với bất kỳ backbone nào khác.
- [x] **Dòng 1 Notebook:** Ghi chính xác `# Model: google/rembert`.
- [x] **Offline Mode:** Load backbone trực tiếp từ Kaggle Models (`aisuko/rembert`), tắt kết nối Internet khi commit.
- [x] **Submission File:** Tệp `submission.csv` gồm đúng 6,000 dòng, 2 cột `id` và `label`.
- [x] **Nhãn hợp lệ:** Các nhãn chuỗi thuộc tập `{'BENIGN', 'JAILBREAK', 'PROMPT_INJECTION'}` (không chứa mã số nguyên 0, 1, 2).
- [x] **Phân phối nhãn:** Tỷ lệ `PROMPT_INJECTION` trong khoảng 5%–10% để tối ưu hóa Macro-F1 metric.
- [x] **Bảo toàn ngôn ngữ:** Giữ nguyên chữ hoa/thường và dấu ký tự Romance (pt, fr, it, en).


In [ ]:
# =============================================================================
# ⚙️ 1. CẤU HÌNH SIÊU THAM SỐ (CHỈ CẦN THAY ĐỔI TẠI CELL NÀY)
# =============================================================================
import json
import os
from pathlib import Path

# Thông tin đội thi & Backbone
TEAM_NAME = "BakhiBachKhoa"
MODEL_NAME = "google/rembert"

# CHẾ ĐỘ CHẠY:
# - SMOKE_TEST = True: Chạy thử nhanh trên Fold 0 (256 mẫu train, 128 val, 64 test, 30 steps) để kiểm tra VRAM và đo ETA.
# - SMOKE_TEST = False: Chạy đầy đủ 5-fold khi Save Version (Commit).
SMOKE_TEST = False

# CHẾ ĐỘ TẠO SUBMISSION:
# - "fold_avg": Trung bình xác suất từ 5 checkpoint của cùng backbone RemBERT (khuyên dùng).
# - "best_fold": Chỉ sử dụng 1 checkpoint duy nhất có điểm validation Macro-F1 cao nhất.
SUBMIT_MODE = "fold_avg"

# Siêu tham số tối ưu cho RemBERT (32 layers, 575M params) - Version 2
BATCH_SIZE = 8          # Batch size khuyến nghị của BTC cho RemBERT
GRAD_ACCUM = 2          # Effective batch size = 8 * 2 = 16
LR = 1.5e-5             # Base Learning Rate cho classifier head / pooler
LLRD = 0.965            # Nới từ 0.95 lên 0.965 (Giúp các layer sâu thích nghi tốt hơn với tiếng Romance)
EPOCHS = 4              # Tăng từ 3 lên 4 (Vì cả 5 fold v1 đều peak ở Epoch 3 và chưa bão hòa)
WARMUP_RATIO = 0.1      # Warmup 10% tổng số optimizer steps
WEIGHT_DECAY = 0.01     # AdamW weight decay (0.0 cho bias và LayerNorm)
ADAM_EPS = 1e-6         # Epsilon cho AdamW tránh lỗi số học FP16
MAX_LEN = 512           # Head 254 + [SEP] + Tail 254 tokens (+ [CLS]/[SEP] = 511 <= 512)
CLASS_WEIGHTS = [1.0, 2.0, 8.8]  # Trọng số loss giải quyết mất cân bằng lớp (PROMPT_INJECTION ~7%)
SEED = 42

# Phân bổ GPU: T4 x2 (GPU 0 chạy fold 0, 2, 4 | GPU 1 chạy fold 1, 3)
# Nếu hệ thống chỉ có 1 GPU, pipeline sẽ tự động chạy tuần tự cả 5 fold trên GPU 0.
GPU_FOLDS = {"0": [0, 2, 4], "1": [1, 3]}

# Thư mục xuất dữ liệu trên Kaggle & tùy chọn Resume
OUT_DIR = "/kaggle/working"
RESUME_DIR = None       # Đặt "/kaggle/input/<ten-output-cu>" nếu cần resume từ phiên bị ngắt
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
Path(f"{OUT_DIR}/logs").mkdir(parents=True, exist_ok=True)

# Ghi toàn bộ cấu hình ra config.json để train_rembert.py đọc
CONFIG_DATA = {
    "TEAM_NAME": TEAM_NAME,
    "MODEL_NAME": MODEL_NAME,
    "SMOKE_TEST": SMOKE_TEST,
    "SUBMIT_MODE": SUBMIT_MODE,
    "BATCH_SIZE": BATCH_SIZE,
    "GRAD_ACCUM": GRAD_ACCUM,
    "LR": LR,
    "LLRD": LLRD,
    "EPOCHS": EPOCHS,
    "WARMUP_RATIO": WARMUP_RATIO,
    "WEIGHT_DECAY": WEIGHT_DECAY,
    "ADAM_EPS": ADAM_EPS,
    "MAX_LEN": MAX_LEN,
    "CLASS_WEIGHTS": CLASS_WEIGHTS,
    "SEED": SEED,
    "GPU_FOLDS": GPU_FOLDS,
    "OUT_DIR": OUT_DIR,
    "RESUME_DIR": RESUME_DIR,
}

config_path = Path(OUT_DIR) / "config.json"
with open(config_path, "w", encoding="utf-8") as f:
    json.dump(CONFIG_DATA, f, indent=2, ensure_ascii=False)

print(f"[*] Cấu hình đã được ghi vào: {config_path}", flush=True)
print(f"[*] Backbone: {MODEL_NAME} | Submit Mode: {SUBMIT_MODE}", flush=True)
print(f"[*] SMOKE_TEST: {SMOKE_TEST} | Batch: {BATCH_SIZE}x{GRAD_ACCUM}=16 | LR: {LR} | LLRD: {LLRD}", flush=True)


In [ ]:
%%writefile train_rembert.py
"""
train_rembert.py
Script huấn luyện 5-Fold chuyên biệt cho google/rembert trên Kaggle GPU T4.
Tuân thủ nghiêm ngặt luật Single-Model (K-Fold checkpoint averaging của DUY NHẤT RemBERT).
"""

import argparse
import gc
import json
import math
import os
import random
import re
import shutil
import sys
import time
import unicodedata
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from sklearn.metrics import f1_score
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Sampler
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    get_cosine_schedule_with_warmup,
)

# Chế độ chạy Offline hoàn toàn theo luật Kaggle
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
torch.backends.cudnn.benchmark = False

# Hằng số chuẩn bài toán
VALID_LABELS = ("BENIGN", "JAILBREAK", "PROMPT_INJECTION")
LABEL2ID = {"BENIGN": 0, "JAILBREAK": 1, "PROMPT_INJECTION": 2}
ID2LABEL = {0: "BENIGN", 1: "JAILBREAK", 2: "PROMPT_INJECTION"}


def set_seed(seed: int) -> None:
    """Cố định seed cho random, numpy và torch trên cả CPU lẫn CUDA."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def log_msg(msg: str, log_file: Optional[Path] = None) -> None:
    """In log ra stdout có flush realtime và lưu tệp log nếu được chỉ định."""
    print(msg, flush=True)
    if log_file is not None:
        try:
            with open(log_file, "a", encoding="utf-8") as f:
                f.write(msg + "\n")
        except Exception:
            pass


def find_model_dir() -> Path:
    """
    Tìm thư mục chứa model RemBERT trên Kaggle Models (/kaggle/input/...).
    Đảm bảo chạy offline không phụ thuộc internet và đúng luật BTC.
    """
    search_dirs = [Path("/kaggle/input"), Path("models"), Path(".")]
    found_configs = []
    for base in search_dirs:
        if base.exists():
            for cfg in base.rglob("config.json"):
                found_configs.append(str(cfg))
                if "rembert" in str(cfg).lower():
                    return cfg.parent
                try:
                    data = json.loads(cfg.read_text(encoding="utf-8"))
                    if data.get("model_type") == "rembert":
                        return cfg.parent
                except Exception:
                    pass
    raise FileNotFoundError(
        "Không tìm thấy model RemBERT! Vui lòng Add Input: "
        "Models -> aisuko/rembert -> Transformers -> default. "
        f"Các config.json tìm thấy: {found_configs}"
    )


def find_file(name: str) -> Path:
    """Tìm file dữ liệu trong /kaggle/input hoặc các thư mục dữ liệu cục bộ."""
    search_dirs = [Path("/kaggle/input"), Path("data"), Path(".")]
    for base in search_dirs:
        if base.exists():
            for f in base.rglob(name):
                if f.is_file():
                    return f
    raise FileNotFoundError(f"Không tìm thấy tệp bắt buộc: '{name}'. Vui lòng kiểm tra lại Input!")


def clean_text_safety(text: Any) -> str:
    """
    Chuẩn hóa văn bản an toàn cho tấn công đa ngôn ngữ (pt, fr, it, en).
    BẢO TỒN CHỮ HOA/THƯỜNG (preserve_case=True) và DẤU ROMANCE.
    """
    if not isinstance(text, str):
        if pd.isna(text) or text is None:
            return ""
        text = str(text)
    # Chuẩn hóa Unicode NFC (giữ nguyên các ký tự có dấu Romance)
    text = unicodedata.normalize("NFC", text)
    # Loại bỏ null byte có thể gây lỗi parser
    text = text.replace("\x00", "")
    # Chuẩn hóa xuống dòng
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    return text.strip()


def prepare_input_ids(texts: List[str], tokenizer: Any) -> List[List[int]]:
    """
    Tokenize hàng loạt và áp dụng Head 254 + [SEP] + Tail 254 Token Truncation.
    Token [SEP] ở giữa thông báo cho Transformer biết đây là hai phân đoạn tách biệt
    (Opening Preamble vs Closing Payload), loại bỏ bigram giả ở mối nối.
    Tổng độ dài tối đa luôn <= 512 token (bao gồm [CLS] và [SEP]).
    """
    cls_id = tokenizer.cls_token_id
    sep_id = tokenizer.sep_token_id
    pad_id = tokenizer.pad_token_id
    assert cls_id is not None, "Tokenizer cls_token_id không được là None"
    assert sep_id is not None, "Tokenizer sep_token_id không được là None"
    assert pad_id is not None, "Tokenizer pad_token_id không được là None"

    enc = tokenizer(texts, add_special_tokens=False, truncation=False)["input_ids"]
    all_ids = []
    for t in enc:
        if len(t) <= 510:
            ids = [cls_id] + t + [sep_id]
        else:
            ids = [cls_id] + t[:254] + [sep_id] + t[-254:] + [sep_id]
        all_ids.append(ids)
    return all_ids


class SmartBatchSampler(Sampler):
    """
    Smart Batching:
    Trộn ngẫu nhiên -> chia mega-batch (bs * 64) -> sort theo độ dài bên trong
    -> chia mini-batch -> trộn thứ tự mini-batch. Giúp dynamic padding đạt hiệu năng cao nhất.
    """
    def __init__(
        self,
        lengths: List[int],
        batch_size: int,
        mega_batch_mult: int = 64,
        shuffle: bool = True,
        seed: int = 42,
    ):
        super().__init__()
        self.lengths = lengths
        self.batch_size = batch_size
        self.mega_batch_size = batch_size * mega_batch_mult
        self.shuffle = shuffle
        self.seed = seed
        self.epoch = 0

    def __iter__(self):
        rng = np.random.RandomState(self.seed + self.epoch)
        indices = np.arange(len(self.lengths))
        if self.shuffle:
            rng.shuffle(indices)

        batches = []
        for i in range(0, len(indices), self.mega_batch_size):
            mega_chunk = indices[i:i + self.mega_batch_size]
            sorted_mega = [int(x) for x in sorted(mega_chunk, key=lambda idx: self.lengths[int(idx)])]
            for j in range(0, len(sorted_mega), self.batch_size):
                batches.append(sorted_mega[j:j + self.batch_size])

        if self.shuffle:
            batch_order = rng.permutation(len(batches))
            batches = [batches[k] for k in batch_order]

        for b in batches:
            yield b

    def __len__(self) -> int:
        return (len(self.lengths) + self.batch_size - 1) // self.batch_size

    def set_epoch(self, epoch: int) -> None:
        self.epoch = epoch


class TextDataset(Dataset):
    """Dataset lưu trữ token ids và nhãn tương ứng."""
    def __init__(self, input_ids: List[List[int]], labels: Optional[List[int]] = None):
        self.input_ids = input_ids
        self.labels = labels

    def __len__(self) -> int:
        return len(self.input_ids)

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        item = {"input_ids": self.input_ids[idx]}
        if self.labels is not None:
            item["label"] = self.labels[idx]
        return item


def make_collate_fn(pad_id: int):
    """Collate function thực hiện dynamic padding tới độ dài lớn nhất trong batch."""
    def collate_fn(batch: List[Dict[str, Any]]) -> Dict[str, torch.Tensor]:
        batch_len = max(len(x["input_ids"]) for x in batch)
        bs = len(batch)
        input_ids = torch.full((bs, batch_len), pad_id, dtype=torch.long)
        attention_mask = torch.zeros((bs, batch_len), dtype=torch.long)
        has_labels = "label" in batch[0]
        labels = []

        for i, item in enumerate(batch):
            seq = item["input_ids"]
            seq_len = len(seq)
            input_ids[i, :seq_len] = torch.tensor(seq, dtype=torch.long)
            attention_mask[i, :seq_len] = 1
            if has_labels:
                labels.append(item["label"])

        res = {"input_ids": input_ids, "attention_mask": attention_mask}
        if has_labels:
            res["labels"] = torch.tensor(labels, dtype=torch.long)
        return res
    return collate_fn


def get_optimizer_grouped_parameters(
    model: nn.Module,
    base_lr: float,
    llrd_rate: float,
    weight_decay: float,
) -> List[Dict[str, Any]]:
    """
    Thiết lập Layer-wise Learning Rate Decay (LLRD) cho RemBERT (32 transformer layers):
    - rembert.embeddings.*: layer id 0
    - rembert.encoder.embedding_hidden_mapping_in.*: layer id 0 (Linear 256->1152 đặc thù RemBERT)
    - rembert.encoder.layer.{i}.*: layer id i + 1 (i = 0..31 -> 1..32)
    - rembert.pooler.*, classifier.*: layer id 33 (base_lr)
    - lr(layer_id) = base_lr * (llrd_rate ** (33 - layer_id))
    - Bias và LayerNorm.weight: weight_decay = 0.0
    - Assert: tổng số params được phân nhóm == sum(p.numel() for p in model.parameters()).
    """
    no_decay_keywords = ("bias", "layernorm.weight", "layer_norm.weight")
    groups: Dict[Tuple[int, float, float], List[nn.Parameter]] = {}

    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue

        if name.startswith("rembert.embeddings.") or name.startswith("rembert.encoder.embedding_hidden_mapping_in."):
            layer_id = 0
        elif name.startswith("rembert.encoder.layer."):
            match = re.search(r"rembert\.encoder\.layer\.(\d+)\.", name)
            if not match:
                raise ValueError(f"Không trích xuất được chỉ số layer từ: {name}")
            layer_id = int(match.group(1)) + 1
            assert 1 <= layer_id <= 32, f"Layer ID ngoài phạm vi 1..32: {layer_id} ({name})"
        elif name.startswith("rembert.pooler.") or name.startswith("classifier."):
            layer_id = 33
        else:
            raise ValueError(f"Tham số không thuộc cấu trúc chuẩn của RemBERT LLRD: {name}")

        is_no_decay = any(nd in name.lower() for nd in no_decay_keywords)
        decay_val = 0.0 if is_no_decay else weight_decay
        lr_val = base_lr * (llrd_rate ** (33 - layer_id))

        group_key = (layer_id, decay_val, lr_val)
        if group_key not in groups:
            groups[group_key] = []
        groups[group_key].append(param)

    param_groups = []
    total_grouped = 0
    for (layer_id, decay_val, lr_val), params in groups.items():
        param_groups.append({
            "params": params,
            "lr": lr_val,
            "weight_decay": decay_val,
        })
        total_grouped += sum(p.numel() for p in params)

    total_model_params = sum(p.numel() for p in model.parameters())
    assert total_grouped == total_model_params, (
        f"LLRD Param count mismatch: grouped={total_grouped} vs model={total_model_params}"
    )
    return param_groups


@torch.no_grad()
def predict_dataset(
    model: nn.Module,
    input_ids_list: List[List[int]],
    pad_id: int,
    batch_size: int = 16,
    device: torch.device = torch.device("cuda"),
) -> np.ndarray:
    """
    Dự đoán xác suất cho tập dữ liệu:
    Sort theo độ dài để batch nhanh, sau đó KHÔI PHỤC đúng thứ tự gốc trước khi trả về.
    """
    model.eval()
    n_samples = len(input_ids_list)
    sorted_indices = sorted(range(n_samples), key=lambda i: len(input_ids_list[i]))

    sorted_probs_list = []
    for start_idx in range(0, n_samples, batch_size):
        chunk_indices = sorted_indices[start_idx:start_idx + batch_size]
        chunk_seqs = [input_ids_list[i] for i in chunk_indices]
        max_len = max(len(seq) for seq in chunk_seqs)
        bs = len(chunk_seqs)

        # Cấp phát trên CPU rồi chuyển cả batch lên GPU 1 lần duy nhất
        input_ids_cpu = torch.full((bs, max_len), pad_id, dtype=torch.long)
        attention_mask_cpu = torch.zeros((bs, max_len), dtype=torch.long)
        for row, seq in enumerate(chunk_seqs):
            seq_len = len(seq)
            input_ids_cpu[row, :seq_len] = torch.tensor(seq, dtype=torch.long)
            attention_mask_cpu[row, :seq_len] = 1

        input_ids = input_ids_cpu.to(device, non_blocking=True)
        attention_mask = attention_mask_cpu.to(device, non_blocking=True)

        with torch.autocast("cuda", dtype=torch.float16, enabled=(device.type == "cuda")):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            probs = torch.softmax(logits.float(), dim=-1).cpu().numpy()
        sorted_probs_list.append(probs)

    sorted_probs = np.vstack(sorted_probs_list)
    orig_probs = np.zeros_like(sorted_probs, dtype=np.float32)
    orig_probs[sorted_indices] = sorted_probs
    assert orig_probs.shape == (n_samples, 3), f"Predict shape mismatch: {orig_probs.shape} vs ({n_samples}, 3)"
    return orig_probs


def train_single_fold(
    fold: int,
    config: Dict[str, Any],
    train_df: pd.DataFrame,
    test_df: pd.DataFrame,
    train_input_ids: List[List[int]],
    test_input_ids: List[List[int]],
    model_dir: Path,
    pad_id: int,
    device: torch.device,
    is_smoke: bool = False,
    out_dir: Path = Path("/kaggle/working"),
    log_file: Optional[Path] = None,
) -> None:
    """Huấn luyện và đánh giá trên một fold cụ thể."""
    log_msg(f"\n{'='*70}\n>>> [BẮT ĐẦU FOLD {fold}] {'(SMOKE TEST)' if is_smoke else ''}\n{'='*70}", log_file)

    # Cố định seed + fold (Đặc tả C10) và reset bộ đếm VRAM đỉnh
    fold_seed = int(config["SEED"]) + int(fold)
    set_seed(fold_seed)
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(device)

    # Kiểm tra resume nếu fold đã hoàn thành
    oof_save_path = out_dir / f"oof_fold{fold}.npy"
    test_save_path = out_dir / f"test_fold{fold}.npy"
    val_idx_save_path = out_dir / f"val_idx_fold{fold}.npy"
    metrics_save_path = out_dir / f"metrics_fold{fold}.json"

    if not is_smoke and oof_save_path.exists() and test_save_path.exists():
        log_msg(f"[*] Fold {fold} đã có file output hoàn chỉnh từ trước. BỎ QUA (Resume)!", log_file)
        return

    # Tách train và val theo fold (val_indices luôn theo thứ tự tăng dần)
    val_mask = (train_df["fold"] == fold).values
    val_indices = np.where(val_mask)[0]
    train_indices = np.where(~val_mask)[0]

    if is_smoke:
        # Cắt mẫu nhỏ cho smoke test theo đúng đặc tả B: 256 train, 128 val, 64 test
        train_indices = train_indices[:256]
        val_indices = val_indices[:128]
        test_input_ids_sub = test_input_ids[:64]
    else:
        test_input_ids_sub = test_input_ids

    train_fold_ids = [train_input_ids[i] for i in train_indices]
    train_fold_labels = [int(train_df["label_id"].iloc[i]) for i in train_indices]

    val_fold_ids = [train_input_ids[i] for i in val_indices]
    val_fold_labels = np.array([int(train_df["label_id"].iloc[i]) for i in val_indices], dtype=int)

    # Khởi tạo model từ checkpoint RemBERT
    log_msg(f"[*] Đang tải backbone RemBERT từ: {model_dir} (seed={fold_seed})", log_file)
    model = AutoModelForSequenceClassification.from_pretrained(
        str(model_dir),
        num_labels=3,
        id2label=ID2LABEL,
        label2id=LABEL2ID,
        ignore_mismatched_sizes=True,
    )

    # Kích hoạt Gradient Checkpointing để tiết kiệm VRAM trên T4
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    if hasattr(model.config, "use_cache"):
        model.config.use_cache = False
    model.to(device)

    # DataLoader với SmartBatchSampler
    batch_size = config["BATCH_SIZE"]
    grad_accum = config["GRAD_ACCUM"]
    collate_fn = make_collate_fn(pad_id)

    lengths = [len(seq) for seq in train_fold_ids]
    sampler = SmartBatchSampler(lengths, batch_size=batch_size, shuffle=True, seed=fold_seed)
    train_dataset = TextDataset(train_fold_ids, train_fold_labels)
    train_loader = DataLoader(train_dataset, batch_sampler=sampler, collate_fn=collate_fn, num_workers=0)

    # Tối ưu hóa & LLRD
    base_lr = config["LR"]
    llrd_rate = config["LLRD"]
    weight_decay = config["WEIGHT_DECAY"]
    param_groups = get_optimizer_grouped_parameters(model, base_lr, llrd_rate, weight_decay)

    optimizer = torch.optim.AdamW(param_groups, eps=config["ADAM_EPS"])

    epochs = config["EPOCHS"]
    steps_per_epoch = math.ceil(len(train_loader) / grad_accum)
    total_optimizer_steps = steps_per_epoch * epochs
    warmup_steps = int(total_optimizer_steps * config["WARMUP_RATIO"])

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_optimizer_steps,
    )

    scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))
    class_weights_tensor = torch.tensor(config["CLASS_WEIGHTS"], dtype=torch.float, device=device)
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor, label_smoothing=0.05)

    # Điểm đánh giá (eval 2 lần/epoch từ epoch 2, và cuối epoch 1; căn chỉnh đúng biên grad_accum)
    eval_schedule = set()
    for ep in range(epochs):
        if ep == 0:
            eval_schedule.add((ep, len(train_loader) - 1))
        else:
            half_opt = max(1, steps_per_epoch // 2)
            mid_pt = min(len(train_loader) - 1, half_opt * grad_accum - 1)
            eval_schedule.add((ep, mid_pt))
            eval_schedule.add((ep, len(train_loader) - 1))

    # Quản lý checkpoint tốt nhất (lưu trong /tmp để không chiếm output Kaggle)
    ckpt_path = Path(f"/tmp/best_fold{fold}.pt")
    if ckpt_path.exists():
        ckpt_path.unlink()

    best_f1 = -1.0
    best_per_class = [0.0, 0.0, 0.0]
    best_step_desc = ""
    n_bad_steps = 0
    t0_fold = time.time()
    step_times = []

    log_msg(
        f"[*] Tổng mẫu train: {len(train_fold_ids):,} | Val: {len(val_fold_ids):,} | "
        f"Steps/epoch: {steps_per_epoch} | Total opt steps: {total_optimizer_steps}",
        log_file,
    )

    # Vòng lặp huấn luyện
    opt_step_count = 0
    stop_smoke = False

    for epoch in range(epochs):
        if stop_smoke:
            break
        sampler.set_epoch(epoch)
        model.train()
        optimizer.zero_grad(set_to_none=True)
        opt_step_start = time.perf_counter()

        for step, batch in enumerate(train_loader):
            is_accum_step = ((step + 1) % grad_accum == 0) or ((step + 1) == len(train_loader))

            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            labels = batch["labels"].to(device, non_blocking=True)

            with torch.autocast("cuda", dtype=torch.float16, enabled=(device.type == "cuda")):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                logits = outputs.logits
                loss = criterion(logits.float(), labels) / grad_accum

            if not torch.isfinite(loss):
                n_bad_steps += 1
                optimizer.zero_grad(set_to_none=True)
                opt_step_start = time.perf_counter()
                continue

            scaler.scale(loss).backward()

            if is_accum_step:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

                scale_before = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                scale_after = scaler.get_scale()

                optimizer.zero_grad(set_to_none=True)

                if scale_after >= scale_before:
                    scheduler.step()
                opt_step_count += 1
                step_times.append(time.perf_counter() - opt_step_start)
                opt_step_start = time.perf_counter()

            # Đánh giá trong chế độ SMOKE_TEST
            if is_smoke and opt_step_count >= 30:
                stop_smoke = True
                break

            # Đánh giá validation theo lịch trình
            if not is_smoke and (epoch, step) in eval_schedule:
                val_probs = predict_dataset(model, val_fold_ids, pad_id, batch_size=16, device=device)
                val_preds = val_probs.argmax(axis=1)
                macro_f1 = float(f1_score(val_fold_labels, val_preds, average="macro", zero_division=0))
                per_class = f1_score(val_fold_labels, val_preds, average=None, zero_division=0).tolist()

                log_msg(
                    f"[Eval Ep {epoch+1} Step {step+1}/{len(train_loader)}] "
                    f"Val Macro-F1: {macro_f1:.4f} | "
                    f"BENIGN: {per_class[0]:.4f} | JAILBREAK: {per_class[1]:.4f} | PROMPT_INJECTION: {per_class[2]:.4f}",
                    log_file,
                )

                if macro_f1 > best_f1:
                    best_f1 = macro_f1
                    best_per_class = per_class
                    best_step_desc = f"epoch_{epoch+1}_step_{step+1}"
                    # Lưu checkpoint nửa chính xác (FP16) trên CPU vào /tmp
                    torch.save(
                        {k: (v.half().cpu() if v.is_floating_point() else v.cpu()) for k, v in model.state_dict().items()},
                        ckpt_path,
                    )

                model.train()
                opt_step_start = time.perf_counter()

    # Báo cáo kết quả Smoke Test (bao gồm kiểm thử predict_dataset trên 128 val + 64 test)
    if is_smoke:
        smoke_val_probs = predict_dataset(model, val_fold_ids, pad_id, batch_size=16, device=device)
        smoke_test_probs = predict_dataset(model, test_input_ids_sub, pad_id, batch_size=16, device=device)
        assert smoke_val_probs.shape == (len(val_fold_ids), 3), "Smoke val shape mismatch"
        assert smoke_test_probs.shape == (len(test_input_ids_sub), 3), "Smoke test shape mismatch"

        peak_vram = torch.cuda.max_memory_allocated(device) / (1024 ** 3) if torch.cuda.is_available() else 0.0
        # Bỏ bước đầu tiên khi tính trung bình tốc độ để tránh chi phí khởi tạo CUDA kernel
        valid_times = step_times[1:] if len(step_times) > 1 else step_times
        sec_per_step = float(np.mean(valid_times)) if valid_times else 0.0
        full_steps_per_fold = math.ceil(11200 / (batch_size * grad_accum)) * epochs
        eta_gpu0_hours = (sec_per_step * full_steps_per_fold * 3) / 3600.0

        log_msg(f"\n{'='*70}", log_file)
        log_msg("[SMOKE TEST THÀNH CÔNG — BÁO CÁO HIỆU NĂNG]", log_file)
        log_msg(f"- VRAM Đỉnh (peak_vram_gb): {peak_vram:.2f} GB (Ngưỡng an toàn T4 16GB: < 14.0 GB)", log_file)
        log_msg(f"- Tốc độ mỗi optimizer step (sec_per_step): {sec_per_step:.3f} giây", log_file)
        log_msg(f"- Số steps mỗi fold (full {epochs} epochs): {full_steps_per_fold}", log_file)
        log_msg(f"- ETA dự kiến cho 3 fold trên GPU 0 (ETA_hours): {eta_gpu0_hours:.2f} giờ", log_file)
        log_msg(f"- Số step lỗi NaN/Inf (n_bad_steps): {n_bad_steps}", log_file)
        log_msg(f"- Kiểm thử Eval/Test Reorder: Val {smoke_val_probs.shape} | Test {smoke_test_probs.shape} OK", log_file)
        log_msg(f"{'='*70}\n", log_file)

        del model, optimizer, scheduler, scaler
        gc.collect()
        torch.cuda.empty_cache()
        return

    # Tải lại checkpoint tốt nhất và dự đoán cho full fold
    log_msg(f"[*] Đang tải checkpoint tốt nhất ({best_step_desc}, Macro-F1: {best_f1:.4f}) từ {ckpt_path}...", log_file)
    if ckpt_path.exists():
        state_dict_half = torch.load(ckpt_path, map_location="cpu", weights_only=True)
        state_dict_float = {k: (v.float() if v.is_floating_point() else v) for k, v in state_dict_half.items()}
        model.load_state_dict(state_dict_float)
    else:
        log_msg("[!] Cảnh báo: Không tìm thấy checkpoint /tmp, sử dụng trọng số hiện tại.", log_file)

    # Dự đoán Out-Of-Fold và Test (Đặc tả C6)
    log_msg(f"[*] Đang sinh OOF probabilities cho Fold {fold}...", log_file)
    oof_probs = predict_dataset(model, val_fold_ids, pad_id, batch_size=16, device=device)
    assert oof_probs.shape[0] == len(val_indices), f"OOF shape mismatch: {oof_probs.shape[0]} vs {len(val_indices)}"

    log_msg(f"[*] Đang sinh Test probabilities cho Fold {fold}...", log_file)
    test_probs = predict_dataset(model, test_input_ids_sub, pad_id, batch_size=16, device=device)
    assert test_probs.shape == (6000, 3), f"Test shape mismatch: got {test_probs.shape}, expected (6000, 3)"

    # Lưu kết quả fold
    np.save(oof_save_path, oof_probs.astype(np.float32))
    np.save(val_idx_save_path, val_indices.astype(np.int64))
    np.save(test_save_path, test_probs.astype(np.float32))

    fold_minutes = (time.time() - t0_fold) / 60.0
    peak_vram = torch.cuda.max_memory_allocated(device) / (1024 ** 3) if torch.cuda.is_available() else 0.0

    fold_metrics = {
        "fold": fold,
        "best_f1": float(best_f1),
        "per_class_f1": [float(x) for x in best_per_class],
        "best_epoch_step": best_step_desc,
        "n_bad_steps": int(n_bad_steps),
        "minutes": float(fold_minutes),
        "peak_vram_gb": float(peak_vram),
    }

    with open(metrics_save_path, "w", encoding="utf-8") as f:
        json.dump(fold_metrics, f, indent=2)

    log_msg(
        f"[HOÀN THÀNH FOLD {fold}] Best F1: {best_f1:.4f} | "
        f"Thời gian: {fold_minutes:.1f} phút | VRAM: {peak_vram:.2f} GB",
        log_file,
    )

    # Dọn dẹp checkpoint và bộ nhớ GPU
    if ckpt_path.exists():
        ckpt_path.unlink()
    del model, optimizer, scheduler, scaler
    gc.collect()
    torch.cuda.empty_cache()


def main():
    parser = argparse.ArgumentParser(description="Huấn luyện google/rembert 5-Fold trên Kaggle.")
    parser.add_argument("--folds", nargs="+", type=int, required=True, help="Danh sách fold cần train (vd: 0 2 4)")
    parser.add_argument("--out_dir", type=str, default="/kaggle/working", help="Thư mục ghi kết quả")
    parser.add_argument("--smoke", action="store_true", help="Chạy chế độ Smoke Test ngắn")
    parser.add_argument("--resume_dir", type=str, default=None, help="Thư mục chứa output version cũ để resume")
    args = parser.parse_args()

    out_dir = Path(args.out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    logs_dir = out_dir / "logs"
    logs_dir.mkdir(parents=True, exist_ok=True)

    # Đọc cấu hình từ config.json
    config_file = out_dir / "config.json"
    if not config_file.exists():
        fallback_cfg = Path("config.json")
        if fallback_cfg.exists():
            config_file = fallback_cfg
        else:
            raise FileNotFoundError(f"Không tìm thấy file cấu hình: {config_file}")

    with open(config_file, "r", encoding="utf-8") as f:
        config = json.load(f)

    # Resume: copy file có sẵn từ resume_dir (ưu tiên CLI arg, sau đó tới config.json)
    resume_dir = args.resume_dir or config.get("RESUME_DIR")
    if resume_dir:
        res_path = Path(resume_dir)
        if res_path.exists():
            log_msg(f"[*] Sao chép checkpoint có sẵn từ resume_dir: {res_path}...")
            for f in res_path.glob("*_fold*.*"):
                dest = out_dir / f.name
                if not dest.exists():
                    shutil.copy2(f, dest)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    log_msg(f"[*] Thiết bị tính toán: {device} | CUDA_VISIBLE_DEVICES: {os.environ.get('CUDA_VISIBLE_DEVICES', 'All')}")

    # Tìm backbone và kiểm tra đúng loại RemBERT
    model_dir = find_model_dir()
    log_msg(f"[*] Đã xác định model RemBERT tại: {model_dir}")
    cfg = AutoConfig.from_pretrained(str(model_dir))
    assert cfg.model_type == "rembert", f"Backbone phải là rembert, phát hiện: {cfg.model_type}"

    # Đọc tokenizer
    log_msg("[*] Đang khởi tạo tokenizer...")
    try:
        tokenizer = AutoTokenizer.from_pretrained(str(model_dir))
    except Exception:
        tokenizer = AutoTokenizer.from_pretrained(str(model_dir), use_fast=False)

    pad_id = tokenizer.pad_token_id
    assert pad_id is not None, "pad_token_id không được là None"

    # Đọc dữ liệu (keep_default_na=False để các chuỗi như 'null'/'NA' không bị biến thành NaN)
    log_msg("[*] Đang đọc train.csv, test.csv và folds.csv...")
    train_path = find_file("train.csv")
    test_path = find_file("test.csv")
    folds_path = find_file("folds.csv")

    train_df = pd.read_csv(train_path, encoding="utf-8-sig", keep_default_na=False, na_values=[""], dtype={"id": str})
    test_df = pd.read_csv(test_path, encoding="utf-8-sig", keep_default_na=False, na_values=[""], dtype={"id": str})
    folds_df = pd.read_csv(folds_path, dtype={"id": str})

    # Merge folds nghiêm ngặt theo ID (validate="1:1"), TUYỆT ĐỐI KHÔNG FALLBACK ÂM THẦM
    train_df = train_df.merge(folds_df, on="id", how="left", validate="1:1")
    assert train_df["fold"].notna().all(), (
        f"LỖI NGHIÊM TRỌNG: folds.csv không khớp với train.csv! Có {train_df['fold'].isna().sum()} dòng NaN."
    )
    train_df["fold"] = train_df["fold"].astype(int)
    train_df["label_id"] = train_df["label"].map(LABEL2ID)

    log_msg(f"[*] Dữ liệu Train: {len(train_df):,} dòng | Test: {len(test_df):,} dòng | Folds: {sorted(train_df['fold'].unique().tolist())}")

    # Chuẩn hóa văn bản và format tiền tố ngôn ngữ
    log_msg("[*] Chuẩn hóa văn bản và định dạng tiền tố 'lang: {language} | {clean_text}'...")
    train_texts = [
        f"lang: {row['language']} | {clean_text_safety(row['text'])}"
        for _, row in train_df.iterrows()
    ]
    test_texts = [
        f"lang: {row['language']} | {clean_text_safety(row['text'])}"
        for _, row in test_df.iterrows()
    ]

    # Tokenize và cắt Head 254 + [SEP] + Tail 254
    log_msg("[*] Tokenize trước toàn bộ tập Train và Test (Head 254 + [SEP] + Tail 254)...")
    train_input_ids = prepare_input_ids(train_texts, tokenizer)
    test_input_ids = prepare_input_ids(test_texts, tokenizer)

    # Chạy lần lượt từng fold được phân công cho process này
    for fold in args.folds:
        fold_log_file = logs_dir / f"fold{fold}.log"
        train_single_fold(
            fold=fold,
            config=config,
            train_df=train_df,
            test_df=test_df,
            train_input_ids=train_input_ids,
            test_input_ids=test_input_ids,
            model_dir=model_dir,
            pad_id=pad_id,
            device=device,
            is_smoke=args.smoke,
            out_dir=out_dir,
            log_file=fold_log_file,
        )


if __name__ == "__main__":
    main()


In [ ]:
# =============================================================================
# 🧪 3. SMOKE TEST (CHẠY THỬ NHANH FOLD 0 TRÊN SESSION TƯƠNG TÁC)
# =============================================================================
import subprocess
import sys

if SMOKE_TEST:
    print("[*] Đang thực thi Smoke Test trên Fold 0 (256 mẫu train, 128 val, 64 test, 30 steps)...", flush=True)
    cmd = [sys.executable, "train_rembert.py", "--folds", "0", "--smoke", "--out_dir", OUT_DIR]
    res = subprocess.run(cmd)
    if res.returncode != 0:
        raise RuntimeError(f"Smoke test thất bại với mã lỗi {res.returncode}! Vui lòng kiểm tra lại log.")
    print("\n[+] SMOKE TEST HOÀN TẤT THÀNH CÔNG!", flush=True)
    print("[+] Kiểm tra các chỉ số ở trên: VRAM đỉnh (<14GB), thời gian mỗi step và ETA.", flush=True)
    print("[+] Để train full 5 fold qua đêm: Đổi SMOKE_TEST = False ở Cell 1 rồi bấm Save Version.", flush=True)
else:
    print("[*] SMOKE_TEST = False. Bỏ qua cell kiểm tra thử để sẵn sàng huấn luyện toàn bộ 5 fold.", flush=True)


In [ ]:
# =============================================================================
# 🚀 4. KHỞI CHẠY HUẤN LUYỆN SONG SONG 2 GPU T4 (HOẶC TUẦN TỰ NẾU 1 GPU)
# =============================================================================
import os
import sys
import time
import subprocess
from pathlib import Path
import torch

if SMOKE_TEST:
    print("[!] SMOKE_TEST = True: Bỏ qua cell khởi chạy chính thức để tránh chạy ngầm.", flush=True)
    print("[!] Hãy kiểm tra kết quả Smoke Test ở Cell 3 trước khi đổi SMOKE_TEST = False.", flush=True)
else:
    n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0
    print(f"[*] Phát hiện {n_gpus} GPU khả dụng trong phiên làm việc.", flush=True)

    log_dir = Path(OUT_DIR) / "logs"
    log_dir.mkdir(parents=True, exist_ok=True)

    processes = []
    log_files = []
    log_paths = []
    extra_args = ["--resume_dir", str(RESUME_DIR)] if RESUME_DIR else []

    if n_gpus >= 2:
        print(f"[*] Chế độ 2 GPU song song theo cấu hình GPU_FOLDS: {GPU_FOLDS}", flush=True)
        for gpu_id, fold_list in GPU_FOLDS.items():
            if not fold_list:
                continue
            folds_str = [str(f) for f in fold_list]
            log_path = log_dir / f"gpu{gpu_id}_launch.log"
            lf = open(log_path, "w", encoding="utf-8", buffering=1)
            log_files.append(lf)
            log_paths.append(log_path)
            env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu_id)}
            cmd = [sys.executable, "train_rembert.py", "--folds", *folds_str, "--out_dir", OUT_DIR, *extra_args]
            p = subprocess.Popen(cmd, env=env, stdout=lf, stderr=subprocess.STDOUT)
            processes.append(p)
            print(f"[+] Đã khởi chạy GPU {gpu_id} (PID: {p.pid}) cho các fold {folds_str}", flush=True)
    else:
        all_folds = sorted({int(f) for flist in GPU_FOLDS.values() for f in flist})
        folds_str = [str(f) for f in all_folds]
        print(f"[*] Chế độ 1 GPU (hoặc CPU): Chạy tuần tự Folds {all_folds} trên GPU 0", flush=True)
        log_path = log_dir / "gpu0_launch.log"
        lf = open(log_path, "w", encoding="utf-8", buffering=1)
        log_files.append(lf)
        log_paths.append(log_path)
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": "0"} if n_gpus == 1 else {**os.environ}
        cmd = [sys.executable, "train_rembert.py", "--folds", *folds_str, "--out_dir", OUT_DIR, *extra_args]
        p = subprocess.Popen(cmd, env=env, stdout=lf, stderr=subprocess.STDOUT)
        processes.append(p)
        print(f"[+] Đã khởi chạy process tuần tự (PID: {p.pid}) cho các fold {folds_str}", flush=True)

    # Vòng lặp theo dõi tiến độ mỗi 120s
    print("[*] Bắt đầu theo dõi tiến độ huấn luyện (cập nhật log mỗi 120 giây)...", flush=True)
    start_time = time.time()
    while any(p.poll() is None for p in processes):
        time.sleep(120)
        elapsed_min = (time.time() - start_time) / 60.0
        print(f"\n--- [Thời gian đã chạy: {elapsed_min:.1f} phút] ---", flush=True)
        for i, (proc, lp) in enumerate(zip(processes, log_paths)):
            status = "Đang chạy" if proc.poll() is None else f"Hoàn thành (mã {proc.poll()})"
            print(f"> Process {i} (PID {proc.pid}) - Trạng thái: {status}", flush=True)
            if lp.exists():
                try:
                    lines = lp.read_text(encoding="utf-8", errors="replace").strip().splitlines()
                    last_3 = "\n  ".join(lines[-3:]) if lines else "  ..."
                    print(f"  {last_3}", flush=True)
                except Exception as e:
                    print(f"  (Chưa đọc được log: {e})", flush=True)
        sys.stdout.flush()

    # Đóng các tệp log
    for lf in log_files:
        lf.close()

    # Kiểm tra mã lỗi trả về của từng process
    for i, (proc, lp) in enumerate(zip(processes, log_paths)):
        if proc.returncode != 0:
            lines = lp.read_text(encoding="utf-8", errors="replace").strip().splitlines()
            last_50 = "\n".join(lines[-50:])
            raise RuntimeError(f"Process {i} thất bại với mã lỗi {proc.returncode}!\n50 dòng log cuối:\n{last_50}")

    print("\n[+] TẤT CẢ CÁC FOLD ĐÃ HUẤN LUYỆN XONG HOÀN HẢO!", flush=True)


In [ ]:
# =============================================================================
# 📊 5. TỔNG HỢP OOF, TỐI ƯU MULTIPLIERS & SINH SUBMISSION CHÍNH THỨC
# =============================================================================
import json
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from sklearn.metrics import f1_score

if SMOKE_TEST:
    print("[!] SMOKE_TEST = True: Bỏ qua cell tổng hợp OOF và sinh submission.", flush=True)
else:
    out_dir = Path(OUT_DIR)
    
    # 1. Đọc dữ liệu Train, Test và Folds
    search_dirs = [Path("/kaggle/input"), Path("data"), Path(".")]
    def find_file_local(name):
        for base in search_dirs:
            if base.exists():
                for f in base.rglob(name):
                    if f.is_file():
                        return f
        raise FileNotFoundError(f"Không tìm thấy {name}")

    train_df = pd.read_csv(find_file_local("train.csv"), encoding="utf-8-sig", keep_default_na=False, na_values=[""], dtype={"id": str})
    test_df = pd.read_csv(find_file_local("test.csv"), encoding="utf-8-sig", keep_default_na=False, na_values=[""], dtype={"id": str})
    folds_df = pd.read_csv(find_file_local("folds.csv"), dtype={"id": str})
    train_df = train_df.merge(folds_df, on="id", how="left", validate="1:1")
    assert train_df["fold"].notna().all(), "folds.csv không khớp 1:1 với train.csv!"

    VALID_LABELS = ("BENIGN", "JAILBREAK", "PROMPT_INJECTION")
    LABEL2ID = {"BENIGN": 0, "JAILBREAK": 1, "PROMPT_INJECTION": 2}
    ID2LABEL = {0: "BENIGN", 1: "JAILBREAK", 2: "PROMPT_INJECTION"}
    y_true = train_df["label"].map(LABEL2ID).values

    # 2. Kiểm tra đầy đủ 5 fold và ghép Out-Of-Fold (OOF) theo đúng val_idx
    missing_folds = [
        k for k in range(5)
        if not ((out_dir / f"oof_fold{k}.npy").exists() and (out_dir / f"val_idx_fold{k}.npy").exists() and (out_dir / f"test_fold{k}.npy").exists())
    ]
    if missing_folds:
        raise RuntimeError(f"Thiếu file kết quả của các fold: {missing_folds}! Kiểm tra lại log huấn luyện.")

    oof_matrix = np.full((len(train_df), 3), np.nan, dtype=np.float32)
    per_fold_metrics = {}

    for k in range(5):
        oof_k = np.load(out_dir / f"oof_fold{k}.npy")
        val_idx_k = np.load(out_dir / f"val_idx_fold{k}.npy")
        oof_matrix[val_idx_k] = oof_k

        metrics_path = out_dir / f"metrics_fold{k}.json"
        if metrics_path.exists():
            with open(metrics_path, "r", encoding="utf-8") as f:
                per_fold_metrics[f"fold_{k}"] = json.load(f)

    # Đảm bảo không còn bất kỳ giá trị NaN nào trong OOF
    assert not np.isnan(oof_matrix).any(), "Ma trận OOF vẫn còn chứa giá trị NaN!"
    print(f"[*] Đã ghép hoàn chỉnh OOF cho toàn bộ {len(train_df):,} mẫu.", flush=True)

    # 3. Tối ưu hóa Class Probability Multipliers trên OOF (Port nguyên logic độc lập)
    def optimize_class_multipliers(y_idx, proba):
        base_preds = proba.argmax(axis=1)
        base_f1 = float(f1_score(y_idx, base_preds, average="macro", zero_division=0))
        best_f1 = base_f1
        best_m = np.ones(3, dtype=float)

        # Grid search 28x28 cho m1, m2 (với m0 cố định bằng 1.0)
        grid_vals = np.linspace(0.3, 3.0, 28)
        for m1 in grid_vals:
            for m2 in grid_vals:
                cand = np.array([1.0, m1, m2], dtype=float)
                preds = (proba * cand).argmax(axis=1)
                score = float(f1_score(y_idx, preds, average="macro", zero_division=0))
                if score > best_f1:
                    best_f1 = score
                    best_m = cand

        # Tinh chỉnh mịn bằng Nelder-Mead
        def _loss(params):
            if np.any(params <= 0.05) or np.any(params > 10.0):
                return 1.0
            m_vec = np.concatenate([[1.0], params])
            preds = (proba * m_vec).argmax(axis=1)
            return -float(f1_score(y_idx, preds, average="macro", zero_division=0))

        res = minimize(_loss, x0=best_m[1:], method="Nelder-Mead", options={"maxiter": 300, "xatol": 1e-3, "fatol": 1e-5})
        if -float(res.fun) > best_f1:
            best_f1 = -float(res.fun)
            best_m = np.concatenate([[1.0], res.x])

        return best_m, base_f1, best_f1

    print("[*] Đang tối ưu hóa Global Class Multipliers trên ma trận OOF 14,000 dòng...", flush=True)
    best_multipliers, global_base_f1, global_tuned_f1 = optimize_class_multipliers(y_true, oof_matrix)
    print(f"[+] Global OOF Macro-F1 (Base -> Global Multipliers): {global_base_f1:.4f} -> {global_tuned_f1:.4f}", flush=True)

    # 4. Tính toán xác suất trên tập Test theo SUBMIT_MODE
    test_fold_probs = [np.load(out_dir / f"test_fold{k}.npy") for k in range(5)]
    for k, p in enumerate(test_fold_probs):
        assert p.shape == (len(test_df), 3), f"Test prob fold {k} có shape sai: {p.shape}"

    if SUBMIT_MODE == "fold_avg":
        print("[*] SUBMIT_MODE = 'fold_avg': Tính trung bình xác suất của 5 checkpoint RemBERT...", flush=True)
        final_test_probs = np.mean(test_fold_probs, axis=0)
    elif SUBMIT_MODE == "best_fold":
        # Chọn fold có validation Macro-F1 cao nhất
        best_fold_idx = max(range(5), key=lambda k: per_fold_metrics.get(f"fold_{k}", {}).get("best_f1", 0.0))
        print(f"[*] SUBMIT_MODE = 'best_fold': Sử dụng checkpoint của Fold {best_fold_idx} (Val F1 cao nhất)...", flush=True)
        final_test_probs = test_fold_probs[best_fold_idx]
    else:
        raise ValueError(f"SUBMIT_MODE không hợp lệ: {SUBMIT_MODE}")

    # 5. Tối ưu hóa Multipliers riêng cho từng ngôn ngữ (Per-Language Multipliers)
    print("\n[*] Đang tối ưu hóa Per-Language Multipliers riêng cho từng ngôn ngữ (pt, fr, it, en)...", flush=True)
    lang_multipliers = {}
    per_language_f1 = {}
    oof_preds_base = oof_matrix.argmax(axis=1)
    oof_preds_tuned = (oof_matrix * best_multipliers).argmax(axis=1)
    final_test_preds = np.zeros(len(test_df), dtype=int)

    for lang in ["pt", "fr", "it", "en"]:
        mask = (train_df["language"] == lang).values
        test_mask = (test_df["language"] == lang).values
        
        y_sub = y_true[mask]
        oof_sub = oof_matrix[mask]
        
        best_m, base_f1, tuned_f1 = optimize_class_multipliers(y_sub, oof_sub)
        lang_multipliers[lang] = best_m
        per_language_f1[lang] = float(tuned_f1)
        oof_preds_tuned[mask] = (oof_sub * best_m).argmax(axis=1)
        print(f"[+] Lang {lang:<2}: Multipliers {np.round(best_m, 3).tolist()} | F1: {base_f1:.4f} -> {tuned_f1:.4f}", flush=True)
        
        # Dự đoán test theo ngôn ngữ tương ứng
        test_sub_probs = final_test_probs[test_mask] * best_m
        final_test_preds[test_mask] = test_sub_probs.argmax(axis=1)

    overall_base_f1 = float(f1_score(y_true, oof_preds_base, average="macro", zero_division=0))
    overall_lang_tuned_f1 = float(f1_score(y_true, oof_preds_tuned, average="macro", zero_division=0))
    cls_f1_base = f1_score(y_true, oof_preds_base, average=None, zero_division=0)
    cls_f1_tuned = f1_score(y_true, oof_preds_tuned, average=None, zero_division=0)

    print(f"\n[+] Tổng hợp 14,000 OOF Macro-F1 (Base):                 {overall_base_f1:.4f}", flush=True)
    print(f"[+] Tổng hợp 14,000 OOF Macro-F1 (Global Multipliers):   {global_tuned_f1:.4f}", flush=True)
    print(f"[+] Tổng hợp 14,000 OOF Macro-F1 (Per-Lang Multipliers): {overall_lang_tuned_f1:.4f} (+{overall_lang_tuned_f1 - overall_base_f1:.4f})", flush=True)
    for idx, name in enumerate(VALID_LABELS):
        print(f"    - F1[{name}]: {cls_f1_base[idx]:.4f} -> {cls_f1_tuned[idx]:.4f}", flush=True)

    # 6. Ghi submission chính thức
    sub_df = pd.DataFrame({"id": test_df["id"], "label": [ID2LABEL[idx] for idx in final_test_preds]})
    sub_df.to_csv(out_dir / "submission.csv", index=False)
    sub_df.to_csv(out_dir / "submission_rembert.csv", index=False)
    np.save(out_dir / "oof_probs_rembert.npy", oof_matrix)
    np.save(out_dir / "test_probs_rembert.npy", final_test_probs)

    # Lưu metrics.json để so sánh và chọn Champion Model
    metrics_summary = {
        "model_name": MODEL_NAME,
        "submit_mode": SUBMIT_MODE,
        "oof_macro_f1_base": float(overall_base_f1),
        "oof_macro_f1_global_tuned": float(global_tuned_f1),
        "oof_macro_f1_tuned": float(overall_lang_tuned_f1),
        "multipliers": best_multipliers.tolist(),
        "lang_multipliers": {k: v.tolist() for k, v in lang_multipliers.items()},
        "per_class_f1_base": {VALID_LABELS[i]: float(cls_f1_base[i]) for i in range(3)},
        "per_class_f1_tuned": {VALID_LABELS[i]: float(cls_f1_tuned[i]) for i in range(3)},
        "per_language_f1": per_language_f1,
        "per_fold": per_fold_metrics,
    }

    with open(out_dir / "metrics.json", "w", encoding="utf-8") as f:
        json.dump(metrics_summary, f, indent=2, ensure_ascii=False)

    print(f"[+] Đã lưu submission.csv ({len(sub_df):,} dòng) và metrics.json thành công!", flush=True)


In [ ]:
# =============================================================================
# 🛡️ 6. KIỂM ĐỊNH CHẤT LƯỢNG SUBMISSION & ĐỐI CHIẾU BASELINE (SANITY CHECK)
# =============================================================================
import json
from pathlib import Path
import numpy as np
import pandas as pd

if SMOKE_TEST:
    print("[!] SMOKE_TEST = True: Bỏ qua sanity check của submission.", flush=True)
else:
    out_dir = Path(OUT_DIR)
    sub_path = out_dir / "submission.csv"
    metrics_path = out_dir / "metrics.json"

    assert MODEL_NAME == "google/rembert", f"Backbone phải là google/rembert, phát hiện: {MODEL_NAME}"
    assert sub_path.exists(), "Tệp submission.csv không tồn tại!"
    sub_df = pd.read_csv(sub_path, dtype={"id": str})

    # 1. Kiểm tra kích thước và cấu trúc cột
    assert len(sub_df) == 6000, f"Số dòng submission phải đúng 6,000, thực tế: {len(sub_df):,}"
    assert list(sub_df.columns) == ["id", "label"], f"Cột submission không đúng chuẩn: {list(sub_df.columns)}"

    # 2. Kiểm tra giá trị rỗng và trùng lặp ID
    assert not sub_df.isna().any().any(), "Submission chứa giá trị rỗng/NaN!"
    assert (sub_df["label"].astype(str).str.len() > 0).all(), "Submission chứa chuỗi rỗng!"
    assert not sub_df["id"].duplicated().any(), "Submission có ID bị trùng lặp!"

    # 3. Kiểm tra nhãn hợp lệ
    VALID_LABELS = {"BENIGN", "JAILBREAK", "PROMPT_INJECTION"}
    unique_labels = set(sub_df["label"].unique())
    invalid_labels = unique_labels - VALID_LABELS
    assert not invalid_labels, f"Phát hiện nhãn không hợp lệ: {invalid_labels}"

    # 4. Kiểm tra tỷ lệ phân phối nhãn
    dist = sub_df["label"].value_counts(normalize=True) * 100
    counts = sub_df["label"].value_counts()
    pi_pct = dist.get("PROMPT_INJECTION", 0.0)

    print("\n" + "="*60, flush=True)
    print("📋 PHÂN PHỐI DỰ ĐOÁN TRÊN TẬP TEST (6,000 DÒNG):", flush=True)
    print("="*60, flush=True)
    for lbl in ["BENIGN", "JAILBREAK", "PROMPT_INJECTION"]:
        cnt = counts.get(lbl, 0)
        pct = dist.get(lbl, 0.0)
        print(f"  - {lbl:<18}: {cnt:>5,} dòng ({pct:>5.2f}%)", flush=True)

    if pi_pct < 4.0 or pi_pct > 12.0:
        print(f"\n[!] CẢNH BÁO: Tỷ lệ PROMPT_INJECTION ({pi_pct:.2f}%) nằm ngoài khoảng kỳ vọng 4%–12%!", flush=True)
    else:
        print(f"\n[+] Tỷ lệ PROMPT_INJECTION ({pi_pct:.2f}%) hoàn toàn đạt chuẩn phân phối mục tiêu (~5%–10%).", flush=True)

    # 5. Đối chiếu OOF Macro-F1 với Baseline 0.9332
    if metrics_path.exists():
        with open(metrics_path, "r", encoding="utf-8") as f:
            metrics = json.load(f)

        oof_f1 = metrics["oof_macro_f1_tuned"]
        baseline_f1 = 0.9332
        diff = oof_f1 - baseline_f1

        print("\n" + "="*60, flush=True)
        print("🏆 KẾT QUẢ ĐỐI CHIẾU VỚI BASELINE (5-FOLD OOF MACRO-F1):", flush=True)
        print("="*60, flush=True)
        print(f"  - Classical ML Baseline OOF Macro-F1:     {baseline_f1:.4f}", flush=True)
        print(f"  - RemBERT 5-Fold OOF (Tuned Multipliers): {oof_f1:.4f} ({'+' if diff >= 0 else ''}{diff:.4f})", flush=True)
        print(f"  - Chi tiết theo ngôn ngữ: {metrics.get('per_language_f1', {})}", flush=True)
        print("="*60, flush=True)

        if diff > 0:
            print("[+] KẾT QUẢ: RemBERT VƯỢT TRỘI BASELINE! Đủ điều kiện cạnh tranh vị trí Champion Model.", flush=True)
        else:
            print("[!] KẾT QUẢ: Điểm OOF chưa vượt qua baseline, cần rà soát lại siêu tham số.", flush=True)

    print("\n[+] TOÀN BỘ CÁC BƯỚC SANITY CHECK ĐÃ HOÀN TẤT THÀNH CÔNG! SẴN SÀNG NỘP KAGGLE.", flush=True)
